# Video Translation Pipeline

**Stack:** faster-whisper (STT + VAD) → Sarvam AI (Translation) → OmniVoice (Voice Cloning TTS) → FFmpeg (Assembly)

**Stages:**
1. Extract audio (24kHz WAV)
2. Transcribe + segment with VAD
3. Translate segments (delimiter-batched)
4. Synthesize each segment with OmniVoice (voice cloned, duration-pinned)
5. Assemble master audio track
6. Merge audio back onto video

> **Runtime:** Set to T4 GPU before running.

In [ ]:
# ── Cell 1: Installs ──────────────────────────────────────────────────────────
# Run once. Restart runtime after this cell completes.

!pip install -q faster-whisper sarvamai pydub

# OmniVoice requires PyTorch with CUDA — Colab T4 already has this.
# Install OmniVoice from source (PyPI release may lag behind).
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git

# FFmpeg is pre-installed on Colab. Verify:
!ffmpeg -version 2>&1 | head -1
!ffprobe -version 2>&1 | head -1

print('\n✅ Installs complete. Restart runtime now if this is your first run.')

In [ ]:
# ── Cell 2: Imports + Config ──────────────────────────────────────────────────

from __future__ import annotations

import os
import re
import subprocess
import tempfile
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torchaudio
from pydub import AudioSegment
from faster_whisper import WhisperModel
from sarvamai import SarvamAI
from google.colab import userdata


@dataclass
class PipelineConfig:
    """
    All pipeline configuration in one place.
    Edit these values before running the pipeline.
    """
    # ── I/O ───────────────────────────────────────────────
    input_video_path: str  = '/content/input.mp4'   # <-- set your video path
    output_dir:       str  = '/content/output'

    # ── Language ──────────────────────────────────────────
    source_language:  str  = 'en-IN'
    target_language:  str  = 'hi-IN'
    translation_mode: str  = 'modern-colloquial'    # or 'formal'

    # ── Audio ─────────────────────────────────────────────
    audio_sample_rate: int = 24_000                 # 24kHz end-to-end

    # ── faster-whisper ────────────────────────────────────
    whisper_model:    str  = 'medium.en'            # CPU-friendly; switch to 'large-v3' on GPU
    whisper_device:   str  = 'cuda'                 # 'cuda' on Colab T4
    whisper_compute:  str  = 'float16'
    vad_min_silence_ms: int = 400                   # silence gap that splits phrases

    # ── Reference chunk for voice cloning ─────────────────
    ref_chunk_min_s: float = 3.0
    ref_chunk_max_s: float = 10.0                   # OmniVoice sweet spot: 3-10s

    # ── OmniVoice ─────────────────────────────────────────
    omnivoice_model:   str  = 'k2-fsa/OmniVoice'
    omnivoice_device:  str  = 'cuda:0'
    omnivoice_dtype: object  = None   # set to torch.float16 in run cell
    omnivoice_num_step: int  = 64                   # 16 = faster, lower quality

    # ── Assembly ──────────────────────────────────────────
    # Max ratio of generated_duration / slot_duration before we truncate.
    # Beyond this, something went wrong — truncate rather than stretch.
    max_duration_ratio: float = 1.25
    room_tone_dbfs:    float  = -40.0               # synthetic gap fill level

    # ── Sarvam ────────────────────────────────────────────
    sarvam_api_key: str = ''                        # set below from Colab secrets

    nestjs_url:    str = ''      # your ngrok URL e.g. https://abc123.ngrok.io
    worker_secret: str = ''      # must match WORKER_SECRET in NestJS .env
    minio_url:     str = ''      # presigned URL — filled per job by NestJS


# ── Load secrets ──────────────────────────────────────────────────────────────
config = PipelineConfig(
    sarvam_api_key=userdata.get('SARVAM_API_KEY'),
)
os.makedirs(config.output_dir, exist_ok=True)

print('Config loaded.')
print(f'  Input  : {config.input_video_path}')
print(f'  Output : {config.output_dir}')
print(f'  Source : {config.source_language} → Target: {config.target_language}')

In [ ]:
# ── Cell 3: Data Classes + PipelineContext ────────────────────────────────────

@dataclass
class Segment:
    """
    One phrase from faster-whisper.
    start / end are in seconds (float).
    translated is populated by TranslateSegmentsStage.
    audio is populated by OmniVoiceTTSStage (torch.Tensor shape (1,T) at 24kHz).
    """
    index:      int
    text:       str
    start:      float
    end:        float
    translated: Optional[str]         = None
    audio:      Optional[torch.Tensor] = None   # shape (1, T), 24kHz

    @property
    def duration(self) -> float:
        """Slot duration in seconds."""
        return round(self.end - self.start, 4)

    @property
    def start_ms(self) -> int:
        return int(self.start * 1000)

    @property
    def end_ms(self) -> int:
        return int(self.end * 1000)


@dataclass
class PipelineContext:
    """
    Carries all state through the pipeline.
    Each stage reads from and writes to this object.
    """
    # Set at init
    input_video_path: str
    config:           PipelineConfig
    work_dir:         str = field(default_factory=lambda: tempfile.mkdtemp(prefix='vidtrans_'))

    # Stage 1 outputs
    audio_path:       Optional[str]   = None
    video_duration:   Optional[float] = None

    # Stage 2 outputs
    segments:         list[Segment]   = field(default_factory=list)
    ref_chunk_path:   Optional[str]   = None    # trimmed WAV for OmniVoice ref
    ref_chunk_text:   Optional[str]   = None    # transcript of the ref chunk

    # Stage 5 output
    master_audio_path: Optional[str]  = None

    # Stage 6 output
    output_video_path: Optional[str]  = None

    # Diagnostics
    _timings: dict = field(default_factory=dict)

    subtitle_en_path: Optional[str] = None
    subtitle_hi_path: Optional[str] = None

    def tick(self, stage: str) -> None:
        self._timings[stage] = time.time()

    def elapsed(self, stage: str) -> float:
        return round(time.time() - self._timings.get(stage, time.time()), 2)


# ── Custom exceptions ─────────────────────────────────────────────────────────

class SilentAudioError(RuntimeError):
    """Audio has no detectable speech. Fail fast — do not retry."""

class TranscriptTooShortError(RuntimeError):
    """Transcript is too short to translate meaningfully. Fail fast."""

class NoValidReferenceChunkError(RuntimeError):
    """Could not find a 3-10s speech-dense chunk for voice cloning. Fail fast."""


print('Data classes defined.')

In [ ]:
# ── Cell 4: Stage 1 — ExtractAudioStage ──────────────────────────────────────

class ExtractAudioStage:
    """
    Stage 1: Extract audio from video as 24kHz mono WAV.

    Why 24kHz:
      - OmniVoice outputs at 24kHz — reference audio should match.
      - faster-whisper resamples internally, so 24kHz input is fine.
      - No resampling needed anywhere in the pipeline.

    Outputs set on ctx:
      ctx.audio_path      — path to extracted WAV
      ctx.video_duration  — duration in seconds (float)
    """

    STAGE = 'stage_1_extract_audio'

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print('\n[Stage 1/6] Extracting audio (FFmpeg → 24kHz mono WAV)…')

        out = os.path.join(ctx.work_dir, 'extracted_audio.wav')

        # Extract audio: 24kHz mono PCM
        cmd = [
            'ffmpeg', '-y',
            '-i', ctx.input_video_path,
            '-vn',
            '-acodec', 'pcm_s16le',
            '-ar', str(ctx.config.audio_sample_rate),
            '-ac', '1',
            out,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f'[Stage 1] FFmpeg failed:\n{result.stderr}')
        if not os.path.exists(out) or os.path.getsize(out) == 0:
            raise RuntimeError('[Stage 1] Output WAV is empty.')

        # Probe video duration
        probe = subprocess.run([
            'ffprobe', '-v', 'error',
            '-show_entries', 'format=duration',
            '-of', 'default=noprint_wrappers=1:nokey=1',
            ctx.input_video_path,
        ], capture_output=True, text=True)

        if probe.returncode != 0 or not probe.stdout.strip():
            raise RuntimeError(f'[Stage 1] ffprobe failed:\n{probe.stderr}')

        ctx.audio_path    = out
        ctx.video_duration = float(probe.stdout.strip())

        size_mb = os.path.getsize(out) / (1024 * 1024)
        print(f'  → {out}  ({size_mb:.1f} MB, {ctx.elapsed(self.STAGE)}s)')
        print(f'  Video duration: {ctx.video_duration:.2f}s')
        print(f'  Sample rate: {ctx.config.audio_sample_rate}Hz')
        return ctx


print('ExtractAudioStage defined.')

In [ ]:
# ── Cell 5: Stage 2 — TranscribeSegmentStage ─────────────────────────────────

class TranscribeSegmentStage:
    """
    Stage 2: Transcribe audio with faster-whisper.

    - vad_filter=True: Silero VAD built into faster-whisper handles silence
      detection and filters non-speech regions. No separate VAD step needed.
    - Returns phrase-level segments (default — no word_timestamps needed).
    - Each segment becomes one Phrase in the pipeline.

    Fail-fast guards:
      - 0 segments returned → SilentAudioError
      - Total transcript < 10 chars → TranscriptTooShortError

    Reference chunk selection:
      - Finds the best contiguous 3-10s window by maximising the number
        of segment seconds that fall inside it.
      - Extracts that slice as a WAV file (ctx.ref_chunk_path).
      - Joins segment texts within that window (ctx.ref_chunk_text).
        This avoids an internal Whisper re-transcription inside OmniVoice.

    Outputs set on ctx:
      ctx.segments        — list[Segment]
      ctx.ref_chunk_path  — path to reference WAV
      ctx.ref_chunk_text  — transcript of reference chunk
    """

    STAGE = 'stage_2_transcribe'

    def __init__(self, config: PipelineConfig):
        self._config = config
        print('  [Whisper] Loading model… (first call downloads weights)')
        self._model = WhisperModel(
            config.whisper_model,
            device=config.whisper_device,
            compute_type=config.whisper_compute,
        )
        print(f'  [Whisper] Model loaded: {config.whisper_model} on {config.whisper_device}')

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print(f'\n[Stage 2/6] Transcribing ({self._config.whisper_model}, VAD enabled)…')

        raw_segments, info = self._model.transcribe(
            ctx.audio_path,
            language='en',
            vad_filter=True,
            vad_parameters=dict(
                min_silence_duration_ms=self._config.vad_min_silence_ms,
            ),
            word_timestamps=False,
        )

        # Materialise the generator — faster-whisper is lazy
        raw_segments = list(raw_segments)

        # ── Fail fast: silence ────────────────────────────
        if not raw_segments:
            raise SilentAudioError(
                '[Stage 2] VAD found no speech in audio. '
                'Video may be silent or language may not match.'
            )

        # ── Build Segment list ────────────────────────────
        segments = [
            Segment(
                index=i,
                text=seg.text.strip(),
                start=seg.start,
                end=seg.end,
            )
            for i, seg in enumerate(raw_segments)
            if seg.text.strip()  # skip any empty segments
        ]

        # ── Fail fast: too short ──────────────────────────
        total_chars = sum(len(s.text) for s in segments)
        if total_chars < 10:
            raise TranscriptTooShortError(
                f'[Stage 2] Transcript is only {total_chars} chars. '
                'Too short to translate meaningfully.'
            )

        ctx.segments = segments

        # ── Reference chunk ───────────────────────────────
        ref_start, ref_end = self._find_best_ref_window(segments, ctx.video_duration)
        ctx.ref_chunk_path = self._extract_chunk_wav(ctx, ref_start, ref_end)
        ctx.ref_chunk_text = self._chunk_transcript(segments, ref_start, ref_end)

        print(f'  Segments: {len(segments)}')
        print(f'  Total chars: {total_chars}')
        print(f'  Detected language: {info.language} (prob={info.language_probability:.2f})')
        print(f'  Ref chunk: {ref_start:.2f}s – {ref_end:.2f}s  '
              f'({ref_end - ref_start:.2f}s)')
        print(f'  Ref text: "{ctx.ref_chunk_text[:80]}…"')
        print(f'  Elapsed: {ctx.elapsed(self.STAGE)}s')

        # Print segment summary
        print('\n  Segment summary:')
        for seg in segments[:5]:  # show first 5
            print(f'    [{seg.start:.2f}s – {seg.end:.2f}s] ({seg.duration:.2f}s) '
                  f'"{seg.text[:60]}"')
        if len(segments) > 5:
            print(f'    … and {len(segments) - 5} more')

        return ctx

    def _find_best_ref_window(
        self,
        segments: list[Segment],
        video_duration: float,
    ) -> tuple[float, float]:
        """
        Slide a window across the audio and score each candidate by
        how many seconds of speech (segment coverage) it contains.

        Window is anchored at segment start times to avoid slicing
        mid-word. For each segment as anchor, extend forward until
        we hit ref_chunk_max_s. Then check if window is >= ref_chunk_min_s.

        Score = sum of (overlap seconds) for all segments inside window.
        """
        min_s = self._config.ref_chunk_min_s
        max_s = self._config.ref_chunk_max_s

        best_score = -1.0
        best_start = 0.0
        best_end   = min(max_s, video_duration)

        for anchor in segments:
            w_start = anchor.start
            w_end   = min(w_start + max_s, video_duration)

            if w_end - w_start < min_s:
                continue

            # Score: total seconds of segment speech inside this window
            score = 0.0
            for seg in segments:
                overlap_start = max(seg.start, w_start)
                overlap_end   = min(seg.end, w_end)
                if overlap_end > overlap_start:
                    score += overlap_end - overlap_start

            if score > best_score:
                best_score = score
                best_start = w_start
                best_end   = w_end

        if best_score < 0:
            raise NoValidReferenceChunkError(
                f'[Stage 2] No valid {min_s}–{max_s}s reference chunk found. '
                'Audio may be too short or have too little speech.'
            )

        return best_start, best_end

    def _extract_chunk_wav(
        self,
        ctx: PipelineContext,
        start: float,
        end: float,
    ) -> str:
        """FFmpeg trim: extract [start, end] from full audio WAV."""
        out = os.path.join(ctx.work_dir, 'ref_chunk.wav')
        cmd = [
            'ffmpeg', '-y',
            '-i', ctx.audio_path,
            '-ss', str(start),
            '-to', str(end),
            '-c', 'copy',
            out,
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError(f'[Stage 2] Reference chunk extraction failed:\n{result.stderr}')
        return out

    def _chunk_transcript(self, segments: list[Segment], start: float, end: float) -> str:
        """
        Join text of all segments that overlap with [start, end].
        Passed to OmniVoice as ref_text to avoid an internal Whisper call.
        """
        texts = [
            seg.text
            for seg in segments
            if seg.start < end and seg.end > start
        ]
        return ' '.join(texts).strip()

    def unload(self) -> None:
        """
        Release Whisper model from VRAM before loading OmniVoice.
        Call this after Stage 2 completes if VRAM is tight.
        """
        del self._model
        torch.cuda.empty_cache()
        print('  [Whisper] Model unloaded from VRAM.')


print('TranscribeSegmentStage defined.')

In [ ]:
# ── Cell 6: Stage 3 — TranslateSegmentsStage ─────────────────────────────────

class _TranslationModel:
    MAYURA           = 'mayura:v1'
    SARVAM_TRANSLATE = 'sarvam-translate:v1'
    MAYURA_LIMIT     = 1000
    SARVAM_LIMIT     = 2000
    SARVAM_MODES     = {'formal'}   # only these use sarvam-translate


class TranslateSegmentsStage:
    """
    Stage 3: Translate each phrase using Sarvam AI.

    Strategy:
      - Pack all segment texts into one API call using |||SEG_N||| delimiters.
      - If combined payload exceeds char limit, batch at segment boundaries
        (never split a segment across batches — each segment is atomic).
      - Parse response by splitting on the same delimiter pattern.
      - Hard invariant: recovered segment count must equal sent count.

    Model selection:
      - 'formal' mode → sarvam-translate:v1 (2000 char limit)
      - all others   → mayura:v1            (1000 char limit)

    Outputs set on ctx:
      ctx.segments[i].translated  — Hindi translation for each segment
    """

    STAGE = 'stage_3_translate'

    def __init__(self, config: PipelineConfig):
        self._config = config
        self._client = SarvamAI(api_subscription_key=config.sarvam_api_key)

        if config.translation_mode in _TranslationModel.SARVAM_MODES:
            self._model = _TranslationModel.SARVAM_TRANSLATE
            self._limit = _TranslationModel.SARVAM_LIMIT
        else:
            self._model = _TranslationModel.MAYURA
            self._limit = _TranslationModel.MAYURA_LIMIT

        print(f'  [Translate] Model: {self._model} | '
              f'Mode: {config.translation_mode} | '
              f'Char limit: {self._limit}')

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print(f'\n[Stage 3/6] Translating {len(ctx.segments)} segments '
              f'({self._config.source_language} → {self._config.target_language})…')

        texts   = [seg.text for seg in ctx.segments]
        results = self._translate_segments(texts)

        # Hard invariant — mapping must be exact
        if len(results) != len(ctx.segments):
            raise RuntimeError(
                f'[Stage 3] Segment count mismatch after translation. '
                f'Sent {len(ctx.segments)}, recovered {len(results)}. '
                f'Delimiter parsing failed — cannot continue.'
            )

        for seg, translated in zip(ctx.segments, results):
            seg.translated = translated

        print(f'  ✅ All {len(ctx.segments)} segments translated ({ctx.elapsed(self.STAGE)}s)')

        # Show first 3 for sanity check
        print('\n  Sample translations:')
        for seg in ctx.segments[:3]:
            print(f'    EN: "{seg.text[:60]}"')
            print(f'    HI: "{seg.translated[:60]}"')
            print()

        return ctx

    def _translate_segments(self, texts: list[str]) -> list[str]:
        """Pack with delimiters, batch at boundaries, parse response."""
        batches = self._build_segment_batches(texts)
        results = []

        for batch_idx, (batch_texts, batch_original_indices) in enumerate(batches):
            print(f'  Batch {batch_idx + 1}/{len(batches)}: '
                  f'{len(batch_texts)} segments…', end=' ')

            payload = '\n'.join(
                f'|||SEG_{i + 1}||| {t}'
                for i, t in zip(batch_original_indices, batch_texts)
            )

            raw    = self._api_call(payload)
            parsed = re.split(r'\|\|\|\s*SEG_\d+\s*\|\|\|', raw)
            parsed = [p.strip() for p in parsed if p.strip()]

            if len(parsed) != len(batch_texts):
                raise RuntimeError(
                    f'[Stage 3] Batch {batch_idx + 1} parse failed. '
                    f'Expected {len(batch_texts)}, got {len(parsed)}.\n'
                    f'Raw response:\n{raw}'
                )

            print(f'✅')
            results.extend(parsed)

        return results

    def _build_segment_batches(
        self,
        texts: list[str],
    ) -> list[tuple[list[str], list[int]]]:
        """
        Group segments into batches that stay under char limit.
        Each segment is atomic — never split across batches.
        Returns list of (batch_texts, original_indices).
        """
        batches:         list[tuple[list[str], list[int]]] = []
        current_texts:   list[str] = []
        current_indices: list[int] = []
        current_len = 0

        for i, text in enumerate(texts):
            overhead  = len(f'|||SEG_{i + 1}||| \n')
            entry_len = len(text) + overhead

            if current_texts and current_len + entry_len > self._limit:
                batches.append((current_texts, current_indices))
                current_texts   = []
                current_indices = []
                current_len     = 0

            current_texts.append(text)
            current_indices.append(i)
            current_len += entry_len

        if current_texts:
            batches.append((current_texts, current_indices))

        return batches

    def _api_call(self, text: str) -> str:
        result = self._client.text.translate(
            input=text,
            source_language_code=self._config.source_language,
            target_language_code=self._config.target_language,
            mode=self._config.translation_mode,
            model=self._model,
        )
        translated = getattr(result, 'translated_text', None)
        if not translated:
            raise RuntimeError('[Stage 3] Empty response from Sarvam API')
        return translated.strip()


print('TranslateSegmentsStage defined.')

In [ ]:
# ── Cell 7: Stage 4 — OmniVoiceTTSStage ──────────────────────────────────────

class OmniVoiceTTSStage:
    """
    Stage 4: Synthesize each translated segment using OmniVoice voice cloning.

    Key design decisions:
      - Model is loaded ONCE and reused across all segments.
      - duration=segment.duration is passed to OmniVoice so the model
        generates audio that fits the original time slot natively.
        This avoids post-hoc stretching in almost all cases.
      - postprocess_output=True (default) removes hallucinated trailing
        silence from the generated audio.
      - ref_text is passed explicitly (from Stage 2 chunk transcript)
        to avoid OmniVoice making an internal Whisper call per segment.
      - Sanity check: if actual generated duration deviates > max_duration_ratio
        from the slot, the audio is truncated (not stretched — something went
        wrong if this triggers).

    Memory note:
      - Call whisper_stage.unload() before constructing this stage if VRAM is tight.
      - OmniVoice float16 on T4 (16GB) is fine alongside freed Whisper memory.

    Outputs set on ctx:
      ctx.segments[i].audio  — torch.Tensor shape (1, T) at 24kHz
    """

    STAGE       = 'stage_4_tts'
    SAMPLE_RATE = 24_000   # OmniVoice always outputs at 24kHz

    def __init__(self, config: PipelineConfig):
        self._config = config
        print('  [OmniVoice] Loading model… (downloads ~few GB on first run)')

        from omnivoice import OmniVoice
        self._model = OmniVoice.from_pretrained(
            config.omnivoice_model,
            device_map=config.omnivoice_device,
            dtype=config.omnivoice_dtype,
        )
        print(f'  [OmniVoice] Model loaded on {config.omnivoice_device}')

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print(f'\n[Stage 4/6] OmniVoice TTS — {len(ctx.segments)} segments…')
        print(f'  Reference: "{ctx.ref_chunk_text[:60]}…"')
        print(f'  Reference audio: {ctx.ref_chunk_path}')

        failed = []

        for seg in ctx.segments:
            if not seg.translated:
                raise RuntimeError(
                    f'[Stage 4] Segment {seg.index} has no translated text. '
                    f'Run TranslateSegmentsStage first.'
                )

            print(f'  [{seg.index + 1}/{len(ctx.segments)}] '
                  f'slot={seg.duration:.2f}s | '
                  f'"{seg.translated[:50]}"…', end=' ')

            try:
                audio_tensors = self._model.generate(
                    text=seg.translated,
                    ref_audio=ctx.ref_chunk_path,
                    ### Changed here
                    # ref_text=ctx.ref_chunk_text,
                    preprocess_prompt=False,
                    ###
                    duration=seg.duration,
                    num_step=self._config.omnivoice_num_step,
                    postprocess_output=True,
                )

                # generate() returns list[Tensor] — take first (single text input)
                audio = audio_tensors[0]  # shape: (1, T)

                # Sanity check: duration deviation
                # actual_dur = audio.shape[-1] / self.SAMPLE_RATE
                #### THis has been changed
                actual_dur = audio.shape[0] / self.SAMPLE_RATE
                ratio      = actual_dur / seg.duration

                if ratio > self._config.max_duration_ratio:
                    # Truncate to slot — something went wrong
                    max_samples = int(seg.duration * self.SAMPLE_RATE)
                    # audio       = audio[:, :max_samples]
                    #### THis has been changed
                    audio       = audio[:max_samples]
                    actual_dur  = seg.duration
                    print(f'[TRUNCATED ratio={ratio:.2f}]', end=' ')

                seg.audio = audio
                print(f'→ {actual_dur:.2f}s ✅')

            except Exception as exc:
                print(f'→ ❌ FAILED: {exc}')
                failed.append((seg.index, str(exc)))
                seg.audio = None   # will be skipped in assembly

        if failed:
            print(f'\n  ⚠️  {len(failed)} segment(s) failed TTS:')
            for idx, err in failed:
                print(f'    Segment {idx}: {err}')

        successful = sum(1 for seg in ctx.segments if seg.audio is not None)
        print(f'\n  TTS complete: {successful}/{len(ctx.segments)} segments '
              f'({ctx.elapsed(self.STAGE)}s)')

        return ctx


print('OmniVoiceTTSStage defined.')

In [ ]:
# ── Cell 8: Stage 5 — AssembleAudioStage ─────────────────────────────────────

class AssembleAudioStage:
    """
    Stage 5: Place synthesized phrase audio onto a master audio track.

    Process:
      1. Create a silent master track of exactly video_duration seconds at 24kHz.
      2. For each segment with audio, overlay it at segment.start.
         - If audio is shorter than slot: leave the rest as silence (no pad).
           OmniVoice + postprocess_output handles this already.
         - If audio is longer than slot (shouldn't happen after sanity check):
           truncate at slot end.
      3. Fill inter-segment gaps (silence regions) with synthetic room tone
         at room_tone_dbfs. This avoids jarring digital silence between phrases.
      4. Segments with failed TTS (audio=None) result in silence in that slot.
      5. Export master track as WAV.

    All operations in numpy at 24kHz (float32) for precision.
    No pydub involved — keeps things simple and avoids extra conversions.

    Outputs set on ctx:
      ctx.master_audio_path  — path to assembled WAV
    """

    STAGE       = 'stage_5_assemble'
    SAMPLE_RATE = 24_000

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print('\n[Stage 5/6] Assembling master audio track…')

        total_samples = int(ctx.video_duration * self.SAMPLE_RATE)

        # Master track: start with room tone everywhere, overlay speech on top
        # This ensures gaps are never digital silence
        master = self._make_room_tone(total_samples, ctx.config.room_tone_dbfs)

        placed = 0
        for seg in ctx.segments:
            if seg.audio is None:
                print(f'  Segment {seg.index}: skipped (no audio)')
                continue

            print(f'Segment {seg.index}: shape={seg.audio.shape}, dtype={seg.audio.dtype}')
            # Convert tensor (1, T) → 1D numpy float32
            # phrase_np = seg.audio.squeeze(0).cpu().float().numpy()
            phrase_np = seg.audio.astype(np.float32)

            start_sample = int(seg.start * self.SAMPLE_RATE)
            end_sample   = int(seg.end   * self.SAMPLE_RATE)
            slot_samples = end_sample - start_sample

            # Clip phrase to slot if it somehow overflows (safety net)
            phrase_np = phrase_np[:slot_samples]

            # Overlay: phrase replaces room tone in its slot
            end_write = min(start_sample + len(phrase_np), total_samples)
            write_len = end_write - start_sample
            master[start_sample:end_write] = phrase_np[:write_len]

            actual_dur = len(phrase_np) / self.SAMPLE_RATE
            print(f'  Segment {seg.index}: placed at {seg.start:.2f}s '
                  f'(slot={seg.duration:.2f}s, actual={actual_dur:.2f}s)')
            placed += 1

        # Export as 16-bit PCM WAV
        out = os.path.join(ctx.work_dir, 'master_audio.wav')
        master_tensor = torch.from_numpy(master).unsqueeze(0)  # (1, T)

        torchaudio.save(out, master_tensor, self.SAMPLE_RATE)

        ctx.master_audio_path = out
        size_mb = os.path.getsize(out) / (1024 * 1024)
        print(f'\n  Placed {placed}/{len(ctx.segments)} segments')
        print(f'  Master track: {out}  ({size_mb:.1f} MB, '
              f'{ctx.video_duration:.2f}s, {ctx.elapsed(self.STAGE)}s)')
        return ctx

    @staticmethod
    def _make_room_tone(n_samples: int, target_dbfs: float) -> np.ndarray:
        """
        Generate white noise at target_dbfs as room tone.
        target_dbfs = -40 gives barely perceptible background hiss
        that avoids jarring digital silence between phrases.
        """
        # Convert dBFS to linear amplitude
        amplitude = 10 ** (target_dbfs / 20.0)
        noise     = np.random.randn(n_samples).astype(np.float32)
        # Normalise then scale
        noise    /= (np.max(np.abs(noise)) + 1e-8)
        noise    *= amplitude
        return noise


print('AssembleAudioStage defined.')

In [ ]:
# ── New Cell B: Subtitle generation stage ────────────────────────────────────

import datetime

class GenerateSubtitlesStage:
    """
    Stage 6.5 (runs after AssembleAudio, before MergeVideo):
    Generates subtitles from ctx.segments.

    Outputs:
      - /work_dir/subtitles_en.vtt   (English WebVTT)
      - /work_dir/subtitles_hi.vtt   (Hindi WebVTT)
      Both paths stored in ctx for MergeVideoStage to embed.

    VTT format:
      WEBVTT

      00:00:00.000 --> 00:00:06.900
      You know, sometimes all you need is 20 seconds of insane courage.
    """

    STAGE = 'stage_6_subtitles'

    def run(self, ctx: PipelineContext) -> PipelineContext:
        ctx.tick(self.STAGE)
        print('\n[Stage 6/7] Generating subtitles (EN + HI)…')

        en_path = os.path.join(ctx.work_dir, 'subtitles_en.vtt')
        hi_path = os.path.join(ctx.work_dir, 'subtitles_hi.vtt')

        self._write_vtt(ctx.segments, en_path, language='en')
        self._write_vtt(ctx.segments, hi_path, language='hi')

        ctx.subtitle_en_path = en_path
        ctx.subtitle_hi_path = hi_path

        print(f'  EN subtitles → {en_path}')
        print(f'  HI subtitles → {hi_path}')
        print(f'  Elapsed: {ctx.elapsed(self.STAGE)}s')
        return ctx

    def _write_vtt(self, segments: list, path: str, language: str) -> None:
        lines = ['WEBVTT', '']
        for i, seg in enumerate(segments, 1):
            text = seg.text if language == 'en' else (seg.translated or seg.text)
            lines.append(str(i))
            lines.append(f'{self._fmt(seg.start)} --> {self._fmt(seg.end)}')
            lines.append(text)
            lines.append('')
        with open(path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(lines))

    @staticmethod
    def _fmt(seconds: float) -> str:
        """Format seconds as HH:MM:SS.mmm for VTT timestamps."""
        td  = datetime.timedelta(seconds=seconds)
        h   = int(td.total_seconds() // 3600)
        m   = int((td.total_seconds() % 3600) // 60)
        s   = td.total_seconds() % 60
        return f'{h:02d}:{m:02d}:{s:06.3f}'


In [ ]:
        cmd = [
            'ffmpeg', '-y',
            '-i', ctx.input_video_path,    # input 0: video
            '-i', ctx.master_audio_path,   # input 1: audio
            '-i', en_srt,                  # input 2: EN subtitles
            '-i', hi_srt,                  # input 3: HI subtitles
            '-c:v', 'copy',
            '-c:a', 'aac',
            '-c:s', 'mov_text',
            '-map', '0:v:0',
            '-map', '1:a:0',
            '-map', '2:0',
            '-map', '3:0',
            '-metadata:s:s:0', 'language=eng', '-metadata:s:s:0', 'title=English',
            '-metadata:s:s:1', 'language=hin', '-metadata:s:s:1', 'title=Hindi',
            '-disposition:s:0', 'default',   # EN track shown by default
            '-disposition:s:1', '0',
            '-t', str(ctx.video_duration),   # IMPORTANT: cap output at video duration
            out,
        ]

In [ ]:
# ── New Cell D: Updated VideoTranslationPipeline with subtitles ───────────────
# Replace the existing VideoTranslationPipeline class with this one.

class VideoTranslationPipeline:
    def __init__(self, config: PipelineConfig):
        self._config = config

    def run(self) -> PipelineContext:
        ctx = PipelineContext(
            input_video_path=self._config.input_video_path,
            config=self._config,
        )

        print('=' * 60)
        print('  VIDEO TRANSLATION PIPELINE')
        print(f'  {self._config.source_language} → {self._config.target_language}')
        print(f'  Input: {self._config.input_video_path}')
        print('=' * 60)

        t0 = time.time()

        ExtractAudioStage().run(ctx)

        whisper_stage = TranscribeSegmentStage(self._config)
        whisper_stage.run(ctx)
        whisper_stage.unload()

        TranslateSegmentsStage(self._config).run(ctx)
        OmniVoiceTTSStage(self._config).run(ctx)
        AssembleAudioStage().run(ctx)
        GenerateSubtitlesStage().run(ctx)   # ← new
        MergeVideoStage().run(ctx)          # ← now stage 7

        print(f'\n{"=" * 60}')
        print(f'  ✅ Pipeline complete in {round(time.time() - t0, 1)}s')
        print(f'  Output: {ctx.output_video_path}')
        print('=' * 60)

        return ctx

In [ ]:
# ── Cell 11: Upload Video ─────────────────────────────────────────────────────
from google.colab import files

print('Select a video file to upload (MP4, MOV, MKV, AVI)...')
uploaded = files.upload()

if not uploaded:
    raise RuntimeError('No file uploaded — please run this cell again and pick a video.')

input_video_path = list(uploaded.keys())[0]
size_mb = os.path.getsize(input_video_path) / (1024 * 1024)
print(f'\n✅ Uploaded: {input_video_path} ({size_mb:.1f} MB)')

In [ ]:
# ── Cell 12: Run ──────────────────────────────────────────────────────────────
config = PipelineConfig(
    input_video_path=input_video_path,   # ← from upload cell
    output_dir='/content/output',
    source_language='en-IN',
    target_language='hi-IN',
    translation_mode='modern-colloquial',
    omnivoice_dtype=torch.float16,
    sarvam_api_key=userdata.get('SARVAM_API_KEY'),
    nestjs_url    = userdata.get('NESTJS_URL'),
    worker_secret = userdata.get('WORKER_SECRET'),
)

pipeline = VideoTranslationPipeline(config)
ctx      = pipeline.run()

from google.colab import files
print(f'\nDownloading: {ctx.output_video_path}')
files.download(ctx.output_video_path)

In [ ]:
# ── New Cell E: Worker polling loop ──────────────────────────────────────────
# Run this cell to start the Colab worker.
# It polls NestJS every 10s for QUEUED jobs and runs the full pipeline.
# Stop it with the Colab interrupt button (■).

import requests as req_lib   # rename to avoid shadowing

NESTJS_URL    = userdata.get('NESTJS_URL')      # e.g. https://abc123.ngrok.io
WORKER_SECRET = userdata.get('WORKER_SECRET')
POLL_INTERVAL = 10  # seconds
HEADERS       = {'X-Worker-Secret': WORKER_SECRET, 'Content-Type': 'application/json'}

def post_progress(job_id: str, progress: int, stage: str, message: str = '') -> None:
    try:
        req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/progress',
            json={'progress': progress, 'stage': stage, 'message': message},
            headers=HEADERS,
            timeout=10,
        )
    except Exception as e:
        print(f'  [Worker] Progress post failed (non-fatal): {e}')


def run_job(job: dict) -> None:
    job_id = job['jobId']
    print(f'\n{"=" * 60}')
    print(f'  JOB RECEIVED: {job_id}')
    print(f'  {job["sourceLanguage"]} → {job["targetLanguage"]}')
    print(f'{"=" * 60}')

    try:
        # 1. Get presigned download URL for input video
        r = req_lib.get(
            f'{NESTJS_URL}/api/worker/{job_id}/input-url',
            headers=HEADERS, timeout=15,
        )
        r.raise_for_status()
        download_url = r.json()['url']

        # 2. Download input video from MinIO
        post_progress(job_id, 0, 'STARTED', 'Downloading input video')
        input_path = f'/content/job_{job_id}_input{os.path.splitext(job["inputFilename"])[1]}'
        with req_lib.get(download_url, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            with open(input_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)
        print(f'  Downloaded: {input_path}')

        # 3. Build config and run pipeline
        config = PipelineConfig(
            input_video_path = input_path,
            output_dir       = '/content/output',
            source_language  = job['sourceLanguage'],
            target_language  = job['targetLanguage'],
            translation_mode = 'modern-colloquial',
            omnivoice_dtype  = torch.float16,
            sarvam_api_key   = userdata.get('SARVAM_API_KEY'),
        )
        os.makedirs(config.output_dir, exist_ok=True)

        # Wrap pipeline so each stage posts progress to NestJS
        # Stage weights: 10, 20, 30, 60, 80, 85, 95 (cumulative %)
        STAGE_PROGRESS = {
            'stage_1_extract_audio': (5,  'Extracting audio'),
            'stage_2_transcribe':    (15, 'Transcribing'),
            'stage_3_translate':     (30, 'Translating'),
            'stage_4_tts':           (65, 'Synthesizing speech'),
            'stage_5_assemble':      (80, 'Assembling audio'),
            'stage_6_subtitles':     (85, 'Generating subtitles'),
            'stage_7_merge':         (95, 'Merging video'),
        }

        original_tick = PipelineContext.tick
        def patched_tick(self_ctx, stage):
            original_tick(self_ctx, stage)
            if stage in STAGE_PROGRESS:
                pct, msg = STAGE_PROGRESS[stage]
                post_progress(job_id, pct, stage, f'Starting: {msg}')
        PipelineContext.tick = patched_tick

        original_mark_done = PipelineContext.mark_done
        STAGE_DONE_PROGRESS = {
            'stage_1_extract_audio': 10,
            'stage_2_transcribe':    28,
            'stage_3_translate':     45,
            'stage_4_tts':           72,
            'stage_5_assemble':      83,
            'stage_6_subtitles':     88,
            'stage_7_merge':         95,
        }

        def patched_mark_done(self_ctx, stage, **meta):
            original_mark_done(self_ctx, stage, **meta)
            if stage in STAGE_DONE_PROGRESS:
                pct = STAGE_DONE_PROGRESS[stage]
                post_progress(job_id, pct, stage, f'Completed {stage}')

        PipelineContext.mark_done = patched_mark_done

        pipeline = VideoTranslationPipeline(config)
        ctx      = pipeline.run()

        PipelineContext.tick = original_tick  # restore

        # 4. Upload output video to MinIO via presigned PUT
        post_progress(job_id, 95, 'stage_7_merge', 'Uploading output')

        output_s3_key = f'outputs/{job_id}-output.mp4'

        # Get presigned PUT URL for output
        r = req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/output-upload-url',
            json={'s3Key': output_s3_key},
            headers=HEADERS, timeout=15,
        )
        r.raise_for_status()
        put_url = r.json()['url']

        # Upload directly to MinIO
        with open(ctx.output_video_path, 'rb') as f:
            up = req_lib.put(put_url, data=f, timeout=600)
            up.raise_for_status()

        # 5. Upload VTT files (small — POST directly to NestJS which saves to MinIO)
        for lang, vtt_path in [('en', ctx.subtitle_en_output_path),
                                ('hi', ctx.subtitle_hi_output_path)]:
            vtt_key = f'outputs/{job_id}-subtitles-{lang}.vtt'
            r = req_lib.post(
                f'{NESTJS_URL}/api/worker/{job_id}/output-upload-url',
                json={'s3Key': vtt_key},
                headers=HEADERS, timeout=10,
            )
            if r.ok:
                put_url_vtt = r.json()['url']
                with open(vtt_path, 'rb') as f:
                    req_lib.put(put_url_vtt, data=f, timeout=30)

        # 6. Mark complete
        req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/complete',
            json={
                's3OutputKey':    output_s3_key,
                's3SubtitleEnKey': f'outputs/{job_id}-subtitles-en.vtt',
                's3SubtitleHiKey': f'outputs/{job_id}-subtitles-hi.vtt',
            },
            headers=HEADERS, timeout=15,
        )
        print(f'\n✅ Job {job_id} complete')

    except Exception as exc:
        import traceback
        print(f'\n❌ Job {job_id} failed: {exc}')
        traceback.print_exc()
        req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/fail',
            json={'errorMessage': str(exc)},
            headers=HEADERS, timeout=10,
        )
    finally:
        # Clean up downloaded input
        if os.path.exists(input_path):
            os.remove(input_path)


# ── Worker loop ───────────────────────────────────────────────────────────────
print(f'Worker started. Polling {NESTJS_URL} every {POLL_INTERVAL}s…')
print('Stop with the interrupt button (■)\n')

while True:
    try:
        r = req_lib.get(
            f'{NESTJS_URL}/api/worker/next-queued',
            headers=HEADERS,
            timeout=10,
        )
        if r.status_code == 204:
            print('.', end='', flush=True)
        elif r.ok:
            run_job(r.json())
        else:
            print(f'\n[Worker] Unexpected response: {r.status_code} {r.text[:100]}')
    except Exception as e:
        print(f'\n[Worker] Poll error: {e}')

    time.sleep(POLL_INTERVAL)

In [ ]:
# ── Cell E: Worker polling loop ──────────────────────────────────────────────
# Run this cell to start the Colab worker.
# It polls NestJS every 10s for QUEUED jobs and runs the full pipeline.
# Stop it with the Colab interrupt button (■).

import requests as req_lib   # rename to avoid shadowing

NESTJS_URL    = userdata.get('NESTJS_URL')
WORKER_SECRET = userdata.get('WORKER_SECRET')
POLL_INTERVAL = 10  # seconds
HEADERS       = {'X-Worker-Secret': WORKER_SECRET, 'Content-Type': 'application/json'}


def post_progress(job_id: str, progress: int, stage: str, message: str = '') -> None:
    try:
        req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/progress',
            json={'progress': progress, 'stage': stage, 'message': message},
            headers=HEADERS,
            timeout=10,
        )
    except Exception as e:
        print(f'  [Worker] Progress post failed (non-fatal): {e}')


def run_job(job: dict) -> None:
    job_id     = job['jobId']
    input_path = None

    print(f'\n{"=" * 60}')
    print(f'  JOB RECEIVED: {job_id}')
    print(f'  {job["sourceLanguage"]} → {job["targetLanguage"]}')
    print(f'{"=" * 60}')

    try:
        # ── 1. Get presigned download URL ─────────────────────────────────────
        r = req_lib.get(
            f'{NESTJS_URL}/api/worker/{job_id}/input-url',
            headers=HEADERS,
            timeout=15,
        )
        r.raise_for_status()
        download_url = r.json()['url']

        # ── 2. Download input video from S3 ───────────────────────────────────
        post_progress(job_id, 0, 'STARTED', 'Downloading input video')
        ext        = os.path.splitext(job['inputFilename'])[1] or '.mp4'
        input_path = f'/content/job_{job_id}_input{ext}'

        with req_lib.get(download_url, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            with open(input_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8 * 1024 * 1024):
                    f.write(chunk)

        size_mb = os.path.getsize(input_path) / (1024 * 1024)
        print(f'  Downloaded: {input_path} ({size_mb:.1f} MB)')

        # ── 3. Build config ────────────────────────────────────────────────────
        config = PipelineConfig(
            input_video_path = input_path,
            output_dir       = '/content/output',
            source_language  = job['sourceLanguage'],
            target_language  = job['targetLanguage'],
            translation_mode = 'modern-colloquial',
            omnivoice_dtype  = torch.float16,
            sarvam_api_key   = userdata.get('SARVAM_API_KEY'),
            nestjs_url       = NESTJS_URL,
            worker_secret    = WORKER_SECRET,
        )
        os.makedirs(config.output_dir, exist_ok=True)

        # ── 4. Progress mapping ────────────────────────────────────────────────
        # Each stage has (start_pct, done_pct, label).
        # emit_progress is called by pipeline.py at stage start and completion.
        STAGE_PROGRESS = {
            'stage_1_extract_audio': (5,  10, 'Extract audio'),
            'stage_2_transcribe':    (15, 28, 'Transcribe'),
            'stage_3_translate':     (30, 45, 'Translate'),
            'stage_4_tts':           (50, 72, 'Synthesize speech'),
            'stage_5_assemble':      (75, 83, 'Assemble audio'),
            'stage_6_subtitles':     (85, 88, 'Subtitles'),
            'stage_7_merge':         (90, 95, 'Merge video'),
        }

        def progress_callback(progress: int, stage: str, message: str) -> None:
            if stage in STAGE_PROGRESS:
                start_pct, done_pct, label = STAGE_PROGRESS[stage]
                pct = start_pct if 'Starting' in message else done_pct
            else:
                pct = progress  # DONE / FAILED use raw value
            post_progress(job_id, pct, stage, message)

        # ── 5. Run pipeline ────────────────────────────────────────────────────
        # Patch _run_with_context on the instance (not the class) so we can
        # inject the progress_callback into the context before any stage runs.
        # PipelineContext.emit_progress() already calls this callback — no stage
        # code needs to change.
        pipeline = VideoTranslationPipeline(config)

        # Inject progress callback by patching run() on the instance.
        # We create the context here, inject the callback, then pass it through.
        original_run = pipeline.run

        def patched_run():
            # Call original but intercept context creation
            # We need to set callback after context is created inside run()
            # Easiest: subclass won't work in notebook, so patch at context level
            # by temporarily overriding PipelineContext.__post_init__
            original_post_init = PipelineContext.__init__

            def patched_init(self_ctx, *args, **kwargs):
                original_post_init(self_ctx, *args, **kwargs)
                self_ctx.progress_callback = progress_callback

            PipelineContext.__init__ = patched_init
            try:
                result = original_run()
            finally:
                PipelineContext.__init__ = original_post_init  # always restore
            return result

        pipeline.run = patched_run
        ctx = pipeline.run()

        # ── 6. Upload output video to S3 ──────────────────────────────────────
        post_progress(job_id, 96, 'stage_7_merge', 'Uploading translated video')

        output_s3_key = f'outputs/{job_id}-output.mp4'

        r = req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/output-upload-url',
            json={'s3Key': output_s3_key},
            headers=HEADERS,
            timeout=15,
        )
        r.raise_for_status()
        put_url = r.json()['url']

        with open(ctx.output_video_path, 'rb') as f:
            up = req_lib.put(put_url, data=f, timeout=600)
            up.raise_for_status()

        print(f'  Output uploaded → {output_s3_key}')

        # ── 7. Upload VTT subtitle files ───────────────────────────────────────
        post_progress(job_id, 97, 'stage_7_merge', 'Uploading subtitles')

        vtt_files = [
            ('en', getattr(ctx, 'subtitle_en_output_path', None)),
            ('hi', getattr(ctx, 'subtitle_hi_output_path', None)),
        ]

        s3_subtitle_en_key = None
        s3_subtitle_hi_key = None

        for lang, vtt_path in vtt_files:
            if not vtt_path or not os.path.exists(vtt_path):
                print(f'  ⚠️  VTT {lang} not found, skipping')
                continue

            vtt_key = f'outputs/{job_id}-subtitles-{lang}.vtt'

            r = req_lib.post(
                f'{NESTJS_URL}/api/worker/{job_id}/output-upload-url',
                json={'s3Key': vtt_key},
                headers=HEADERS,
                timeout=10,
            )
            if not r.ok:
                print(f'  ⚠️  Could not get presigned URL for {vtt_key}: {r.status_code}')
                continue

            put_url_vtt = r.json()['url']
            with open(vtt_path, 'rb') as f:
                up = req_lib.put(put_url_vtt, data=f, timeout=30)
                if up.ok:
                    print(f'  VTT {lang} uploaded → {vtt_key}')
                    if lang == 'en':
                        s3_subtitle_en_key = vtt_key
                    else:
                        s3_subtitle_hi_key = vtt_key
                else:
                    print(f'  ⚠️  VTT {lang} upload failed: {up.status_code}')

        # ── 8. Mark job complete ───────────────────────────────────────────────
        complete_payload = {'s3OutputKey': output_s3_key}
        if s3_subtitle_en_key:
            complete_payload['s3SubtitleEnKey'] = s3_subtitle_en_key
        if s3_subtitle_hi_key:
            complete_payload['s3SubtitleHiKey'] = s3_subtitle_hi_key

        req_lib.post(
            f'{NESTJS_URL}/api/worker/{job_id}/complete',
            json=complete_payload,
            headers=HEADERS,
            timeout=15,
        )

        post_progress(job_id, 100, 'DONE', 'Translation complete')
        print(f'\n✅ Job {job_id} complete')

    except Exception as exc:
        import traceback
        print(f'\n❌ Job {job_id} failed: {exc}')
        traceback.print_exc()

        try:
            req_lib.post(
                f'{NESTJS_URL}/api/worker/{job_id}/fail',
                json={'errorMessage': str(exc)},
                headers=HEADERS,
                timeout=10,
            )
        except Exception:
            pass  # Don't let fail-reporting crash the loop

    finally:
        # Clean up downloaded input regardless of success or failure
        if input_path and os.path.exists(input_path):
            os.remove(input_path)
            print(f'  Cleaned up: {input_path}')


# ── Worker loop ───────────────────────────────────────────────────────────────
print(f'Worker started. Polling {NESTJS_URL} every {POLL_INTERVAL}s…')
print('Stop with the interrupt button (■)\n')

while True:
    try:
        r = req_lib.get(
            f'{NESTJS_URL}/api/worker/next-queued',
            headers=HEADERS,
            timeout=10,
        )
        if r.status_code == 204:
            print('.', end='', flush=True)
        elif r.ok:
            job = r.json()
            if job:   # guard against empty body on 200
                run_job(job)
        else:
            print(f'\n[Worker] Unexpected response: {r.status_code} {r.text[:200]}')
    except Exception as e:
        print(f'\n[Worker] Poll error: {e}')

    time.sleep(POLL_INTERVAL)

In [ ]:
# ── Cell 13: Debug Utilities (optional) ──────────────────────────────────────
# Run individual stages or inspect intermediate results.

# ── Listen to a single synthesized segment ────────────────────────────────────
from IPython.display import Audio, display

def play_segment(ctx: PipelineContext, index: int) -> None:
    """Play the synthesized audio for a specific segment index."""
    seg = next((s for s in ctx.segments if s.index == index), None)
    if seg is None:
        print(f'Segment {index} not found')
        return
    if seg.audio is None:
        print(f'Segment {index} has no audio (TTS failed)')
        return

    audio_np = seg.audio.astype(np.float32)
    print(f'Segment {index}: [{seg.start:.2f}s – {seg.end:.2f}s]')
    print(f'  EN: {seg.text}')
    print(f'  HI: {seg.translated}')
    print(f'  Duration: slot={seg.duration:.2f}s  actual={len(audio_np)/24000:.2f}s')
    display(Audio(audio_np, rate=24000))


def play_master(ctx: PipelineContext) -> None:
    """Play the assembled master audio track."""
    if not ctx.master_audio_path:
        print('Master audio not yet assembled')
        return
    display(Audio(ctx.master_audio_path))


def print_segments(ctx: PipelineContext) -> None:
    """Print all segments with translation and timing info."""
    print(f'Total segments: {len(ctx.segments)}')
    print(f'Video duration: {ctx.video_duration:.2f}s')
    print()
    for seg in ctx.segments:
        status = '✅' if seg.audio is not None else '❌'
        print(f'{status} [{seg.index}] {seg.start:.2f}s–{seg.end:.2f}s '
              f'({seg.duration:.2f}s)')
        print(f'     EN: {seg.text}')
        print(f'     HI: {seg.translated or "(not translated)"}')
        if seg.audio is not None:
            actual = seg.audio.shape[-1] / 24000
            print(f'     Audio: {actual:.2f}s generated')
        print()


# Example usage (run after pipeline.run()):
print_segments(ctx)
play_segment(ctx, 0)
play_master(ctx)

print('Debug utilities defined.')